# 🧠 Time-to-Failure Prediction — 1D Convolutional Neural Network

**Dataset:** LANL Earthquake Prediction (laboratory acoustic emission signals)
**Goal:** Predict `time_to_failure` directly from the **raw** `acoustic_data` waveform — no manual
statistical feature engineering.
**Model:** 1D CNN (`tensorflow.keras`)

---

## Scientific Motivation

Notebook 2 compressed every 150,000-sample acoustic segment into 7 hand-chosen statistics (`mean`,
`std`, percentiles, extrema) before handing it to XGBoost. That compression is a modeling *choice*,
not a law of nature — it assumes those 7 numbers capture everything relevant about the precursor
signal. This notebook removes that assumption entirely: a 1D convolutional network consumes the
**unabridged waveform** and learns its own hierarchical representations — local spike morphology,
frequency content, temporal patterning — directly from the raw signal, the same way a CNN over pixels
learns its own visual features rather than relying on hand-crafted edge detectors.

The two notebooks together form a controlled comparison: identical segments, identical train/test
split, identical evaluation metric — the only variable is whether the model sees engineered
statistics or the raw waveform.

### Notebook outline
1. Imports & configuration
2. Stream raw signal & build tensors
3. Exploratory waveform visualization
4. Train / test split & signal normalization
5. Build the 1D-CNN architecture
6. Train the model
7. Evaluation on held-out test set
8. Training curves
9. Persist the trained model (native Keras v3 `.keras` format)
10. Result summary


## 1 · Imports & Configuration

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow import keras
from tensorflow.keras import layers
from tqdm.keras import TqdmCallback

sys.path.append(os.path.abspath(os.path.join("..", "src")))
from config import (
    LANL_TRAIN_PATH, SEGMENT_SIZE, LANL_TEST_SIZE, CNN_CONFIG,
    RANDOM_STATE, MODELS_DIR, FIGURES_DIR, PLOT_COLORS, PLOT_DPI, PLOT_STYLE,
)
from logging_utils import console, get_logger, metrics_table, section
from preprocessing import build_segment_tensor_table

logger = get_logger(__name__)

# Shared dark, high-contrast publication theme (identical to src/train_cnn.py)
plt.style.use(PLOT_STYLE)
sns.set_style("darkgrid", {"axes.facecolor": "#1b1f27", "grid.color": PLOT_COLORS["grid"]})
plt.rcParams.update({
    "figure.dpi": PLOT_DPI,
    "savefig.dpi": PLOT_DPI,
    "axes.edgecolor": PLOT_COLORS["text"],
    "axes.labelcolor": PLOT_COLORS["text"],
    "text.color": PLOT_COLORS["text"],
    "xtick.color": PLOT_COLORS["text"],
    "ytick.color": PLOT_COLORS["text"],
    "font.size": 11,
})

np.random.seed(RANDOM_STATE)
keras.utils.set_random_seed(RANDOM_STATE)
console.print(f"[bold]Segment size:[/bold] {SEGMENT_SIZE:,} samples  |  [bold]Random seed:[/bold] {RANDOM_STATE}")

## 2 · Stream the Raw Signal & Build Tensors

Unlike the XGBoost pipeline, the CNN consumes the **full raw waveform** of every 150,000-sample
segment, reshaped into a `(150000, 1)` tensor. The network learns its own internal representations
directly from the signal rather than relying on hand-crafted statistics.

> ⚠️ **Memory note:** raw-tensor segments are far larger than the 7-value statistical feature rows
> used for XGBoost. For very large datasets, consider a `tf.data` generator pipeline instead of
> materializing the full array in memory, as is done here for clarity.

In [ ]:
section("Load Raw Acoustic Segments as Tensors")

_READER_DTYPES = {"acoustic_data": np.int16, "time_to_failure": np.float64}


def load_raw_tensors():
    reader = pd.read_csv(LANL_TRAIN_PATH, dtype=_READER_DTYPES, chunksize=SEGMENT_SIZE)

    X_chunks, y_chunks = [], []
    leftover_acoustic = np.array([], dtype=np.int16)
    leftover_ttf = np.array([], dtype=np.float64)

    for chunk in reader:
        acoustic = np.concatenate([leftover_acoustic, chunk["acoustic_data"].values])
        ttf = np.concatenate([leftover_ttf, chunk["time_to_failure"].values])

        n_full = len(acoustic) // SEGMENT_SIZE
        usable = n_full * SEGMENT_SIZE

        if n_full > 0:
            # build_segment_tensor_table renders its own tqdm progress bar
            # over segments within this chunk
            X_seg, y_seg = build_segment_tensor_table(
                acoustic[:usable], ttf[:usable], segment_size=SEGMENT_SIZE
            )
            X_chunks.append(X_seg)
            y_chunks.append(y_seg)

        leftover_acoustic = acoustic[usable:]
        leftover_ttf = ttf[usable:]

    X = np.concatenate(X_chunks, axis=0)
    y = np.concatenate(y_chunks, axis=0)
    return X, y


X, y = load_raw_tensors()
console.print(f"[green]X shape:[/green] {X.shape}  |  [green]y shape:[/green] {y.shape}")

## 3 · Exploratory Waveform Visualization

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
colors = [PLOT_COLORS["primary"], PLOT_COLORS["secondary"]]

for ax, idx, color in zip(axes, [0, min(5, X.shape[0] - 1)], colors):
    ax.plot(X[idx, :, 0], linewidth=0.4, color=color)
    ax.set_title(
        f"Raw Acoustic Waveform — Segment {idx} (time_to_failure \u2248 {y[idx]:.2f}s)",
        fontweight="bold",
    )
    ax.set_ylabel("Amplitude")
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Sample index within segment")
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "cnn_sample_waveforms.png"), dpi=PLOT_DPI, bbox_inches="tight")
plt.show()

**Reading the waveform:** large discrete amplitude spikes against an otherwise quiet baseline are
visible micro-fracture events — exactly the precursor behavior the statistical descriptors in
Notebook 2 (`std`, `p01`, `p99`) were designed to summarize. Here, the CNN's convolutional filters get
to discover that pattern (and any subtler ones a fixed 7-statistic summary would miss) on their
own.

## 4 · Train / Test Split & Signal Normalization

In [ ]:
section("Train / Test Split & Normalization")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=LANL_TEST_SIZE, random_state=RANDOM_STATE
)

# Z-score normalize using TRAIN statistics only, to avoid test-set leakage
train_mean = float(X_train.mean())
train_std = float(X_train.std())

X_train = (X_train - train_mean) / train_std
X_test = (X_test - train_mean) / train_std

console.print(f"Train: [bold]{X_train.shape}[/bold]  |  Test: [bold]{X_test.shape}[/bold]")
console.print(f"Normalization stats -> mean: {train_mean:.4f}, std: {train_std:.4f}")

## 5 · Build the 1D-CNN Architecture

| # | Layer | Configuration |
|---|---|---|
| 1 | `Conv1D` | 16 filters, kernel size 10, ReLU |
| 2 | `MaxPooling1D` | pool size 10 |
| 3 | `Conv1D` | 32 filters, kernel size 10, ReLU |
| 4 | `MaxPooling1D` | pool size 10 |
| 5 | `Flatten` | — |
| 6 | `Dense` | 64 units, ReLU |
| 7 | `Dropout` | rate 0.3 |
| 8 | `Dense` | 1 unit, linear (regression output) |

Compiled with the **Adam** optimizer and **MAE** loss. All values are sourced from the project-wide
`config.CNN_CONFIG`, so the architecture here is guaranteed to match `src/train_cnn.py` exactly.

In [ ]:
section("Build the 1D-CNN Architecture")

def build_cnn_model(input_shape, cfg):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv1D(cfg["conv1_filters"], cfg["conv1_kernel"], activation="relu"),
        layers.MaxPooling1D(cfg["pool1_size"]),
        layers.Conv1D(cfg["conv2_filters"], cfg["conv2_kernel"], activation="relu"),
        layers.MaxPooling1D(cfg["pool2_size"]),
        layers.Flatten(),
        layers.Dense(cfg["dense_units"], activation="relu"),
        layers.Dropout(cfg["dropout_rate"]),
        layers.Dense(1, activation="linear"),
    ])
    model.compile(optimizer=cfg["optimizer"], loss=cfg["loss"], metrics=["mae"])
    return model

model = build_cnn_model(input_shape=(SEGMENT_SIZE, 1), cfg=CNN_CONFIG)
model.summary(print_fn=lambda line: console.print(f"[dim]{line}[/dim]"))

## 6 · Train the Model

In [ ]:
section("Train the Model")
console.print(
    f"Training for [bold]{CNN_CONFIG['epochs']}[/bold] epochs "
    f"(batch_size=[bold]{CNN_CONFIG['batch_size']}[/bold])..."
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=CNN_CONFIG["epochs"],
    batch_size=CNN_CONFIG["batch_size"],
    verbose=0,
    callbacks=[TqdmCallback(verbose=1)],  # rich-friendly tqdm progress bar in place of Keras' default verbose output
)

## 7 · Evaluation on Held-Out Test Set

In [ ]:
section("Evaluation on Held-Out Test Set")

y_pred = model.predict(X_test, verbose=0).flatten()

metrics = {
    "mae": mean_absolute_error(y_test, y_pred),
    "rmse": float(np.sqrt(mean_squared_error(y_test, y_pred))),
    "r2": r2_score(y_test, y_pred),
}
console.print(metrics_table("1D-CNN -- Test Set Metrics", metrics))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test, y_pred, alpha=0.45, s=14, color=PLOT_COLORS["tertiary"], edgecolors="none")
lims = [float(y_test.min()), float(y_test.max())]
ax.plot(lims, lims, "--", linewidth=1.8, color=PLOT_COLORS["accent"], label="Perfect prediction")
ax.set_xlabel("Actual time_to_failure (s)")
ax.set_ylabel("Predicted time_to_failure (s)")
ax.set_title("1D-CNN: Actual vs. Predicted Time-to-Failure", fontweight="bold")
ax.legend(frameon=False)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "cnn_actual_vs_predicted.png"), dpi=PLOT_DPI, bbox_inches="tight")
plt.show()

## 8 · Training Curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(history.history["loss"], label="Train MAE", color=PLOT_COLORS["primary"], linewidth=2)
ax.plot(history.history["val_loss"], label="Validation MAE", color=PLOT_COLORS["secondary"], linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("MAE Loss")
ax.set_title("1D-CNN Training Curve", fontweight="bold")
ax.legend(frameon=False)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "cnn_training_curve.png"), dpi=PLOT_DPI, bbox_inches="tight")
plt.show()

## 9 · Persist the Trained Model

> **Format note:** the model is saved in the **native Keras v3 format (`.keras`)**, not the legacy
> HDF5 (`.h5`) format used in early prototypes of this project. `.h5` is deprecated in current
> TensorFlow/Keras and was found to fail on reload in this environment's Keras version — `.keras`
> is the format `keras.Model.save()` now recommends by default.

In [ ]:
section("Persist Model Artifact")

os.makedirs(MODELS_DIR, exist_ok=True)
model_path = os.path.join(MODELS_DIR, "cnn_1d_lanl.keras")  # native Keras v3 format, NOT legacy .h5
model.save(model_path)
console.print(f"[bold green]Model saved[/bold green] -> {model_path}")

# Persist normalization stats -- required for consistent inference later
norm_stats = {"mean": train_mean, "std": train_std}
console.print(f"[bold yellow]Normalization stats (required at inference):[/bold yellow] {norm_stats}")

---
## 10 · Result Summary

| Metric | Value |
|---|---|
| MAE | *(see Section 7 metrics table above)* |
| Epochs | 10 |
| Batch Size | 16 |
| Optimizer / Loss | Adam / MAE |
| Train / Test Split | 80 / 20 |
| Saved format | Native Keras v3 (`.keras`) |

By learning its own representations directly from the raw waveform, the 1D-CNN provides a direct,
controlled comparison against the hand-engineered statistical-feature approach in Notebook 2 — the
same segments, the same split, the same evaluation metric, with only the input representation
differing. Compare the MAE values from both notebooks side by side to assess whether end-to-end
representation learning earns its added training cost over lightweight feature engineering for this
acoustic emission regression task.
